# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahaamjad2004/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking / Priority Scoring (Binary Classification Proxy)Although the underlying learning objective can be trained as binary classification (predicting whether a page's trajectory is declining vs. stable/growing), the real operational task is ranking.Editorial teams do not evaluate every positive prediction across a database of 30,000 URLs. Instead, they require an ordered queue of pages sorted by predicted decay probability weighted by traffic exposure, triaging only the top $K$ pages per review cycle.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Task type definition and configuration
TASK_TYPE = "Ranking / Priority Scoring"
PREDICTION_OBJECTIVE = "P(trend_direction == 'down')"
EVALUATION_MODE = "Top-K prioritized queue"

print(f"Task Type: {TASK_TYPE}")
print(f"Model Objective: {PREDICTION_OBJECTIVE}")
print(f"Deployment Format: {EVALUATION_MODE}")


Task Type: Ranking / Priority Scoring
Model Objective: P(trend_direction == 'down')
Deployment Format: Top-K prioritized queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
Target Definition: is_declining = (trend_direction == 'down')

Nature of Target (Ground Truth vs. Proxy): This is an observable proxy label. True content decay includes nuance like search intent shift, semantic obsolescence, or competitor displacement. A quarterly drop in impressions/traffic (trend_direction == 'down') serves as a measurable operational proxy.

Leakage Guardrail: trend_pct is the underlying percentage calculation used to construct trend_direction. Therefore, trend_pct represents the outcome in disguise and must be strictly excluded from all training feature matrices.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

# Standard repo environment path handling
if "google.colab" in sys.modules:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target proxy column
df["target_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Inspect distribution
target_dist = df["target_declining"].value_counts(normalize=True).rename({0: "Stable/Up (0)", 1: "Declining (1)"})
print("Target Proxy Class Distribution:")
print(target_dist.round(3))


Target Proxy Class Distribution:
target_declining
Declining (1)    0.542
Stable/Up (0)    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Primary Metric: Precision@K ($K=20, 50, 100$)Why: Content operations teams have fixed labor capacity (e.g., refreshing 20–50 pages per sprint). Precision@K directly quantifies what percentage of their allocated time was spent on pages genuinely suffering traffic decline.Secondary Metric: Lift over BaselineWhy: The dataset has a natural decline prevalence of ~54.2%. A ranking mechanism is only valuable if its Precision@50 significantly exceeds this random baseline.Why ROC-AUC or Raw Accuracy is Misleading: Global accuracy treats errors on obscure, zero-impression pages the same as errors on high-value organic landing pages. Precision@K focuses strictly on the decision boundary where editorial money is spent.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Random baseline vs baseline heuristic
y_true = df["target_declining"].values
random_scores = np.random.RandomState(42).rand(len(df))

base_precision_50 = precision_at_k(random_scores, y_true, k=50)
print(f"Random Baseline Precision@50: {base_precision_50:.3f}")
print(f"Dataset Overall Prevalence:    {y_true.mean():.3f}")

Random Baseline Precision@50: 0.560
Dataset Overall Prevalence:    0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
Unit of Analysis: One URL / page observed over a 90-day evaluation window.

Each row represents a single piece of published content characterized exclusively by pre-decision signals (historical impressions, position tiers, CTR, and days since update) alongside the target proxy.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_signals = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

unit_of_analysis_df = df[feature_signals + ["target_declining"]].head(5)
print("Unit of Analysis Dataframe Sample (1 row = 1 page):")
unit_of_analysis_df


Unit of Analysis Dataframe Sample (1 row = 1 page):


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,target_declining
0,187,20,3803,10.6,0.76,3221.0,1
1,445,25,15320,20.3,0.05,2481.0,1
2,141,20,12581,36.5,0.09,3515.0,1
3,463,22,11751,6.2,0.49,NaN,0
4,263,14,19140,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed heuristic (such as days_since_last_update >= 180 and impressions_90d >= 500) creates brittle, non-compensatory cutoffs:Plateau and Tie-Breaking Failures: A simple rule creates wide blocks of tied scores, losing discriminatory power as soon as you evaluate deeper into the queue ($K > 20$).Multi-Factor Interaction: A page with low CTR and mediocre SERP position (15–25) may decay much faster than an authoritative pillar page untouched for a year. Fixed rules fail to weigh position drift against update latency.Non-Linear Thresholding: Machine learning models capture continuous probability surfaces across search exposure, identifying decay risks across varied page types without manual calibration.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

X = df[feature_signals].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["target_declining"]

# Depth-2 model
clf = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
clf.fit(X, y)
tree_scores = clf.predict_proba(X)[:, 1]

# Fixed hand rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_scores = stale * visible * df["impressions_90d"]

print("Comparison at Depth:")
for k in [20, 50, 100]:
    hr_k = precision_at_k(hand_scores, y, k)
    tr_k = precision_at_k(tree_scores, y, k)
    print(f"Precision@{k:3d} -> Hand Rule: {hr_k:.3f} | Model: {tr_k:.3f}")


Comparison at Depth:
Precision@ 20 -> Hand Rule: 0.900 | Model: 0.550
Precision@ 50 -> Hand Rule: 0.680 | Model: 0.600
Precision@100 -> Hand Rule: 0.630 | Model: 0.660


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.